# Notebook 11 — Project Evaluation
## AI-Powered FIFA World Cup 2026 Fantasy Intelligence Platform

**Author:** Preynsh Thukral  
**Date:** June 2026  
**Status:** Final Presentation Notebook

---

This notebook is the **capstone summary** of the entire WC2026 Fantasy Intelligence project.
It does not re-train models or re-run the pipeline — it loads final outputs and presents results.

### Notebook Structure
1. Project Overview
2. Dataset Summary
3. Feature Engineering Summary
4. Model Performance
5. SHAP Insights
6. Tournament Context Impact
7. Top Players
8. Hidden Gems
9. Captain Recommendations
10. Squad Optimization Results
11. Key Findings
12. Limitations
13. Future Work

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
from pathlib import Path
from IPython.display import display, Markdown

# Paths
PROCESSED = Path("../data/processed")
MODELS    = Path("../models")
REPORTS   = Path("../reports")
REPORTS.mkdir(exist_ok=True)

# Style
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans"
})

print("✅ Libraries loaded.")

---
## 1. Project Overview

### Problem Statement
FIFA World Cup 2026 is the first 48-team World Cup, creating an expanded player pool of ~1,248 players across 48 nations. Fantasy managers face a significantly harder selection problem compared to previous tournaments.

### What This System Does
| Component | Description |
|-----------|-------------|
| **Data Engineering** | Merged 1.88M player-match records from Transfermarkt (2012–2026) |
| **Feature Engineering** | 9 rolling form features + 8 Understat advanced metrics |
| **XGBoost Model** | Predicts expected fantasy points per match |
| **WC2026 Inference** | Maps 929/1,248 squad players to Transfermarkt; generates predictions |
| **Tournament Context** | Adjusts projections using Elo-based fixture difficulty multipliers |
| **Recommendation Engine** | Ranks players; flags hidden gems, safe captains, aggressive captains |
| **Squad Optimizer** | Integer linear programming under budget and positional constraints |

### Pipeline Summary
```
Raw Data (Transfermarkt + Understat + WC2026 Squads + Elo + Fantasy Prices)
    ↓ Notebook 02: Data Warehouse Build
    ↓ Notebook 03: Fantasy Scoring Validation
    ↓ Notebook 04: Feature Engineering (rolling windows)
    ↓ Notebook 05: XGBoost Model Training
    ↓ Notebook 06: Recommendation Engine
    ↓ Notebook 07: WC2026 Squad Inference
    ↓ Notebook 08: Fixture Difficulty Engine (Elo)
    ↓ Notebook 09: SHAP Explainability
    ↓ Notebook 10: Multi-Objective Squad Optimizer
```

---
## 2. Dataset Summary

In [ ]:
# Load the model dataset (full history)
model_df = pd.read_csv(PROCESSED / "model_dataset_final.csv")
model_df["date"] = pd.to_datetime(model_df["date"])

print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"Total player-match records:   {len(model_df):>12,}")
print(f"Unique players:               {model_df['player_id'].nunique():>12,}")
print(f"Unique matches:               {model_df['game_id'].nunique():>12,}")
print(f"Date range:                   {model_df['date'].min().date()} → {model_df['date'].max().date()}")
print(f"Feature columns:              {model_df.shape[1]}")
print()

# Chronological split recap
train = model_df[model_df["date"] < "2023-01-01"]
val   = model_df[(model_df["date"] >= "2023-01-01") & (model_df["date"] < "2024-01-01")]
test  = model_df[model_df["date"] >= "2024-01-01"]

split_data = {
    "Split": ["Train", "Validation", "Test (Holdout)"],
    "Rows": [f"{len(train):,}", f"{len(val):,}", f"{len(test):,}"],
    "Date Range": [
        f"{train['date'].min().date()} → {train['date'].max().date()}",
        f"{val['date'].min().date()} → {val['date'].max().date()}",
        f"{test['date'].min().date()} → {test['date'].max().date()}"
    ],
    "% of Data": [
        f"{len(train)/len(model_df)*100:.1f}%",
        f"{len(val)/len(model_df)*100:.1f}%",
        f"{len(test)/len(model_df)*100:.1f}%"
    ]
}
display(pd.DataFrame(split_data))

In [ ]:
# Data sources breakdown
sources = pd.DataFrame({
    "Source": ["Transfermarkt", "Understat", "WC2026 Squad List", "Elo Ratings", "FIFA Fantasy Prices"],
    "Description": [
        "Player appearances, market values, goals, assists (2012–2026)",
        "Advanced metrics: xG, xA, shots, key passes (2014–2026)",
        "48-nation registered WC2026 squads (1,248 players)",
        "Club/country Elo ratings for fixture difficulty scoring",
        "Official FIFA fantasy point prices for squad budget constraint"
    ],
    "Records": ["1.88M player-match rows", "~620K rows", "1,248 players", "48 national teams", "749 priced players"]
})
display(sources)

---
## 3. Feature Engineering Summary

In [ ]:
features = pd.DataFrame({
    "Feature": [
        "avg_fp_last_3", "avg_fp_last_5",
        "avg_goals_last_5", "avg_assists_last_5",
        "avg_minutes_last_5", "std_fp_last_5",
        "matches_played_last_5",
        "market_value_in_eur", "position"
    ],
    "Type": ["Rolling", "Rolling", "Rolling", "Rolling", "Rolling",
             "Rolling", "Rolling", "Static", "Static"],
    "Window": ["L3", "L5", "L5", "L5", "L5", "L5", "L5", "—", "—"],
    "Description": [
        "Average fantasy points over last 3 appearances",
        "Average fantasy points over last 5 appearances (primary form signal)",
        "Average goals scored over last 5 appearances",
        "Average assists over last 5 appearances",
        "Average minutes played over last 5 appearances (availability proxy)",
        "Std deviation of FP over last 5 (volatility / upside signal)",
        "Count of matches with >0 minutes in last 5 (playing time)",
        "Transfermarkt market value in EUR (proxy for quality tier)",
        "Position (Attack=0, Defender=1, Goalkeeper=2, Midfield=3)"
    ]
})
print("Core Feature Set (9 features):")
display(features)

print("\nAll rolling features use shift(1) before rolling to prevent data leakage.")
print("Cold-start rows (<3 matches in last 5) are excluded from training.")

---
## 4. Model Performance

In [ ]:
# Model comparison table (results from Notebook 05)
results = pd.DataFrame({
    "Model": ["Dummy (Mean Baseline)", "Linear Regression", "Random Forest", "XGBoost (Final)"],
    "RMSE": [2.7198, 2.6805, 2.6453, 2.6473],
    "MAE":  [2.0742, 2.0204, 1.9839, 1.9690],
    "R²":   [-0.0043, 0.0245, 0.0500, 0.0508],
    "Notes": [
        "Predicts training mean for every row",
        "Weak coefficients; avg_minutes dominates",
        "n_estimators=100, max_depth=10, n_jobs=-1",
        "n_estimators=200, max_depth=6, lr=0.1  ← SELECTED"
    ]
})
print("=" * 60)
print("MODEL BENCHMARKING — Test Set (2024-01-01 → 2026-05-24)")
print("=" * 60)
display(results.style.highlight_min(subset=["RMSE", "MAE"], color="lightgreen")
        .highlight_max(subset=["R²"], color="lightgreen"))

print("\nXGBoost Improvement over dummy baseline:")
print(f"  RMSE reduction: {2.7198 - 2.6473:.4f} points  ({(2.7198-2.6473)/2.7198*100:.1f}% better)")
print(f"  MAE  reduction: {2.0742 - 1.9690:.4f} points  ({(2.0742-1.9690)/2.0742*100:.1f}% better)")

In [ ]:
# Feature importance from saved model
model = joblib.load(MODELS / "xgb_fantasy_model_v1.joblib")

FEATURES = [
    "position", "market_value_in_eur",
    "avg_fp_last_3", "avg_fp_last_5",
    "avg_goals_last_5", "avg_assists_last_5",
    "avg_minutes_last_5", "std_fp_last_5", "matches_played_last_5"
]

imp = pd.DataFrame({
    "Feature": FEATURES[:len(model.feature_importances_)],
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v > 0.3 else "#3498db" if v > 0.1 else "#95a5a6" for v in imp["Importance"]]
ax.barh(imp["Feature"], imp["Importance"], color=colors)
ax.set_xlabel("Gain-based Feature Importance")
ax.set_title("XGBoost Feature Importances\n(xgb_fantasy_model_v1)")
ax.axvline(0.1, color="black", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig(REPORTS / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/feature_importance.png")

In [ ]:
# Prediction error distribution
test_preds = pd.read_csv(PROCESSED / "test_predictions.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Error distribution
axes[0].hist(test_preds["prediction_error"], bins=80, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(0, color="red", linewidth=1.5, linestyle="--")
axes[0].axvline(test_preds["prediction_error"].mean(), color="orange", linewidth=1.5, label=f"Mean = {test_preds['prediction_error'].mean():.3f}")
axes[0].set_title("Prediction Error Distribution (Test Set)")
axes[0].set_xlabel("Predicted − Actual (Fantasy Points)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Actual vs Predicted scatter (sample)
sample = test_preds.sample(10000, random_state=42)
axes[1].scatter(sample["actual_fantasy_points"], sample["predicted_fantasy_points"],
                alpha=0.15, s=10, color="steelblue")
lims = [min(sample["actual_fantasy_points"].min(), sample["predicted_fantasy_points"].min()),
        max(sample["actual_fantasy_points"].max(), sample["predicted_fantasy_points"].max())]
axes[1].plot(lims, lims, "r--", lw=1.5, label="Perfect prediction")
axes[1].set_title("Actual vs. Predicted (10k sample)")
axes[1].set_xlabel("Actual Fantasy Points")
axes[1].set_ylabel("Predicted Fantasy Points")
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORTS / "model_error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/model_error_analysis.png")

---
## 5. SHAP Insights

SHAP (SHapley Additive exPlanations) values were computed in **Notebook 09**.

Key findings:
- **`avg_fp_last_5`** is the dominant feature (~56% gain importance). Players with strong recent form carry the largest positive SHAP impact.
- **`avg_minutes_last_5`** is the second most impactful feature — availability is a critical signal. A player injured for 3 of the last 5 games is aggressively penalised.
- **`market_value_in_eur`** and **`position`** together capture player quality tier and role-based scoring patterns.
- **`std_fp_last_5`** (volatility) has a non-linear SHAP profile — moderate volatility is rewarded in the high-points tail.

The model's predictions are transparent and interpretable — each prediction can be decomposed into feature-level contributions.

---
## 6. Tournament Context Impact

In [ ]:
# Compare raw vs adjusted projections
wc_raw = pd.read_csv(PROCESSED / "world_cup_predictions_final.csv")
wc_adj = pd.read_csv(PROCESSED / "world_cup_predictions_adjusted.csv")

# Try to merge on player identifier
id_col = "player_id" if "player_id" in wc_raw.columns else "name"

# Find the projection columns
proj_col_raw = [c for c in wc_raw.columns if "predict" in c.lower() or "project" in c.lower()][0]
proj_col_adj = [c for c in wc_adj.columns if "adjust" in c.lower() or "project" in c.lower()][0]

merged = wc_raw[[id_col, proj_col_raw, "country" if "country" in wc_raw.columns else id_col]].merge(
    wc_adj[[id_col, proj_col_adj]], on=id_col, how="inner"
)
merged["delta"] = merged[proj_col_adj] - merged[proj_col_raw]
merged["delta_pct"] = (merged["delta"] / merged[proj_col_raw].abs()) * 100

print("Tournament Context Adjustment Statistics:")
print(f"  Players with positive adjustment (favourable group): {(merged['delta'] > 0).sum()}")
print(f"  Players with negative adjustment (tough group):      {(merged['delta'] < 0).sum()}")
print(f"  Max upside boost:      {merged['delta'].max():.3f} pts  ({merged['delta_pct'].max():.1f}%)")
print(f"  Max penalty reduction: {merged['delta'].min():.3f} pts  ({merged['delta_pct'].min():.1f}%)")
print(f"  Mean absolute shift:   {merged['delta'].abs().mean():.3f} pts")

# Country-level re-ranking
country_col = "country" if "country" in merged.columns else None
if country_col:
    country_impact = merged.groupby(country_col)["delta"].mean().sort_values(ascending=False)
    print("\nTop 5 nations that benefit most from context adjustment:")
    print(country_impact.head(5).to_string())
    print("\nTop 5 nations most penalised:")
    print(country_impact.tail(5).to_string())

---
## 7. Top Players

In [ ]:
# Load master player pool (post-context, with prices)
pool = pd.read_csv(PROCESSED / "master_player_pool.csv")
pool["price"] = pool["price"].fillna(pool["price"].median())

# Rename adjusted_projection for clarity if needed
proj_col = "adjusted_projection" if "adjusted_projection" in pool.columns else pool.columns[4]

POS_MAP = {0: "FWD", 1: "MID", 2: "GK", 3: "DEF"}
pool["pos_label"] = pool["position"].map(POS_MAP)

print("=" * 60)
print("TOP 10 OVERALL — ADJUSTED PROJECTION")
print("=" * 60)
top10 = pool.nlargest(10, proj_col)[["player", "country", "pos_label", "price", proj_col]]
top10.columns = ["Player", "Country", "Position", "Price (€M)", "Projected FP"]
display(top10.reset_index(drop=True))

print("\nTop 5 by position:")
for pos_num, pos_name in sorted(POS_MAP.items(), key=lambda x: x[1]):
    sub = pool[pool["position"] == pos_num].nlargest(5, proj_col)[["player", "country", "price", proj_col]]
    print(f"\n{pos_name}:")
    display(sub.reset_index(drop=True))

---
## 8. Hidden Gems

In [ ]:
# Hidden gems = high projected output relative to price
gem_col = "gem_score_adj" if "gem_score_adj" in pool.columns else None

if gem_col:
    # Remove players with no price info
    gems = pool[pool["price"].notna()].copy()
    gems = gems[gems[proj_col] > gems[proj_col].quantile(0.5)]  # Only above-median projection
    gems = gems.nlargest(10, gem_col)[["player", "country", "pos_label", "price", proj_col, gem_col]]
    gems.columns = ["Player", "Country", "Position", "Price (€M)", "Proj FP", "Gem Score"]
    
    print("=" * 60)
    print("TOP 10 HIDDEN GEMS")
    print("High output, underpriced — gem_score = percentile(pts) − percentile(price)")
    print("=" * 60)
    display(gems.reset_index(drop=True))
else:
    # Fallback: compute value score manually
    pool["value_score"] = pool[proj_col] / pool["price"]
    gems = pool[pool[proj_col] > pool[proj_col].quantile(0.5)].nlargest(10, "value_score")
    display(gems[["player", "country", "pos_label", "price", proj_col, "value_score"]].reset_index(drop=True))

---
## 9. Captain Recommendations

In [ ]:
pool["std_fp_last_5"] = pool["std_fp_last_5"].fillna(0)
pool["safe_score"]   = pool[proj_col] / (1 + pool["std_fp_last_5"])
pool["agg_score"]    = pool[proj_col] * (1 + pool["std_fp_last_5"])

display_cols = ["player", "country", "pos_label", "price", proj_col, "std_fp_last_5"]

print("🛡️  SAFE CAPTAINS — High floor, low variance (captain score = pts / (1 + volatility))")
safe = pool.nlargest(5, "safe_score")[display_cols]
safe.columns = ["Player", "Country", "Pos", "Price", "Proj FP", "Volatility"]
display(safe.reset_index(drop=True))

print("\n🚀  AGGRESSIVE CAPTAINS — High ceiling, high volatility (score = pts × (1 + volatility))")
agg = pool.nlargest(5, "agg_score")[display_cols]
agg.columns = ["Player", "Country", "Pos", "Price", "Proj FP", "Volatility"]
display(agg.reset_index(drop=True))

if gem_col:
    pool["diff_score"] = pool[proj_col] * (1 + pool[gem_col] / 100)
    print("\n💎  DIFFERENTIAL CAPTAINS — Hidden gems with high output (under £8.5M)")
    diff = pool[pool["price"] <= 8.5].nlargest(5, "diff_score")[display_cols]
    diff.columns = ["Player", "Country", "Pos", "Price", "Proj FP", "Volatility"]
    display(diff.reset_index(drop=True))

---
## 10. Squad Optimization Results

Three squad strategies were optimised using **PuLP integer linear programming** (see Notebook 10).

**Constraints:**
- 15 total players (2 GK, 5 DEF, 5 MID, 3 FWD)
- €100M total budget
- Max 3 players per nation

**Strategies:**
| Strategy | Objective |
|----------|----------|
| META | Maximise raw projected fantasy points |
| HIGH UPSIDE | Maximise `pts × (1 + volatility)` — tournament winner approach |
| DIFFERENTIAL VALUE | Maximise `(pts/price) + gem_score × 0.1` — beat the field |

*Run Notebook 10 for the full squad tables — they depend on live PuLP solver output.*

---
## 11. Key Findings

### What Works Well
1. **Form is king.** `avg_fp_last_5` alone accounts for ~56% of model gain importance. Recent performance is the single strongest predictor of fantasy output.
2. **Availability matters.** `avg_minutes_last_5` is the 2nd most important feature. A player averaging 70+ minutes over their last 5 games is a significantly safer pick.
3. **Tournament context adds signal.** The Elo-based fixture multiplier (range 0.5–1.5×) creates meaningful re-ranking. Players in soft groups (high expected win probability) benefit by up to +50% on projected points.
4. **Multi-objective optimization produces distinct squads.** The three LP strategies consistently yield different players, demonstrating that the gem score and volatility signals carry real information.

### Quantitative Results
| Metric | Value |
|--------|-------|
| Test RMSE (XGBoost) | 2.6473 pts |
| Test MAE (XGBoost) | 1.9690 pts |
| RMSE improvement over dummy | 2.7% |
| WC2026 squad mapping coverage | 74.4% (929 / 1,248) |
| Players with full 5-match history | 95.8% of model rows |
| Fantasy price matching coverage | 97.8% (742 / 749 priced) |

---
## 12. Limitations

1. **Low R² (~5%)** — Individual match-level fantasy points are highly stochastic. Variance is dominated by chance events (injuries, manager decisions, weather, referee) that no historical model can predict. R² measures explained variance on noisy data; the model's value is in ranking players, not pin-pointing exact scores.

2. **Squad mapping gap (25.6% unmapped)** — 319 WC2026 players could not be matched to Transfermarkt IDs, primarily players from smaller nations (Jordan, South Africa, etc.) with limited Transfermarkt coverage. These are all depth players unlikely to fantasy-impact the tournament.

3. **Cold-start problem** — 36 matched players had zero historical data. Median-imputed features were used. These players should be avoided in final squads.

4. **Understat coverage (55%)** — Advanced metrics (xG, xA) were only available for 45.4% of the fantasy-point pool. The v2 dataset supplements these but cannot close the coverage gap fully.

5. **No real-time data** — The model uses pre-tournament snapshots. Player form, injuries, and squad roles will evolve during the tournament. The system should be retrained after each round.

6. **Fantasy price matching** — 7 players remain unmatched to official fantasy prices and receive the median price fallback. These are fringe players unlikely to feature in optimised squads.

---
## 13. Future Work

### Near-Term
- **Chip into the 25.6% mapping gap** using multilingual name normalisation and scraping from additional sources (FIFA official, Wikipedia)
- **Round-by-round retraining** — After each WC round, update rolling features with actual match data and re-run inference + optimizer
- **Streamlit dashboard** — The `app/` directory is reserved for a live fantasy assistant web interface

### Medium-Term
- **Sequence models (LSTM/Transformer)** — Replace rolling window averages with learned sequence representations that capture trend, momentum, and fatigue
- **Multi-task learning** — Jointly predict goals, assists, clean sheets, and minutes to reduce label noise vs. the composite fantasy_points target
- **Bayesian calibration** — Post-hoc calibrate XGBoost output distributions for better uncertainty quantification in the captaincy engine

### Long-Term
- **Generalise beyond WC2026** — The pipeline is designed to be tournament-agnostic. Plug in Euro 2028, Copa America, or Premier League with different squad/price data
- **Automated data refresh** — Connect scrapers to a scheduler (Airflow / GitHub Actions) for continuous data ingestion throughout the tournament
- **Community validation** — Open-source the prediction outputs for community benchmarking against expert analyst picks

In [ ]:
print("=" * 60)
print("PROJECT EVALUATION COMPLETE")
print("=" * 60)
print()
print("Pipeline status:")
pipeline = [
    ("01", "Data Audit",              "✅"),
    ("02", "Data Warehouse Build",    "✅"),
    ("03", "Fantasy Scoring",         "✅"),
    ("04", "Feature Engineering",     "✅"),
    ("05", "Modeling",                "✅"),
    ("06", "Recommendation Engine",   "✅"),
    ("07", "WC2026 Inference",        "✅"),
    ("08", "Fixture Difficulty",      "✅"),
    ("09", "Explainability (SHAP)",   "✅"),
    ("10", "Squad Optimizer",         "✅"),
    ("11", "Project Evaluation",      "✅"),
]
for nb, name, status in pipeline:
    print(f"  {nb} — {name:<30} {status}")

print()
print("Key outputs:")
print("  models/xgb_fantasy_model_v1.joblib")
print("  data/processed/world_cup_predictions_final.csv")
print("  data/processed/world_cup_predictions_adjusted.csv")
print("  data/processed/master_player_pool.csv")
print("  reports/feature_importance.png")
print("  reports/model_error_analysis.png")